In [ ]:
import pandas as pd
import numpy as np

# Load CLSI breakpoints with multi-row header
clsi_df = pd.read_excel("/content/CLSI_breakpoints.xlsx", sheet_name="All combined", header=[0, 1])

# Inspect the column headers (optional for debugging)
#print(clsi_df.columns)

# Flatten multi-index columns
clsi_df.columns = [
    col[0] if col[0] not in ["CLSI MIC breakpoints"] else col[1]
    for col in clsi_df.columns
]

# Rename columns for consistency
clsi_df.rename(columns={
    'Organism name': 'Organism',
    'S': 'S_breakpoint',
    'I': 'I_breakpoint',  # not used, but kept for clarity
    'R': 'R_breakpoint'
}, inplace=True)

# Load Venatorx data
venatorx_df = pd.read_excel("/content/Venatorx surveillance data_2024_06_06.xlsx")

# Antibiotic abbreviation to full name mapping
abbrev_to_name = {
    'CAZ': 'Ceftazidime',
    'C': 'Chloramphenicol',
    'CIP': 'Ciprofloxacin',
    'CL': 'Clindamycin',
    'FEP': 'Cefepime',
    'GM': 'Gentamicin',
    'IPM': 'Imipenem',
    'LVX': 'Levofloxacin',
    'MEM': 'Meropenem',
    'MI': 'Minocycline',
    'SXT': 'Trimethoprim/Sulfamethoxazole',
    'TIM': 'Ticarcillin/Clavulanic acid',
    'TZP': 'Piperacillin/Tazobactam'
}

# MIC interpretation function
def interpret_mic(mic_value, s_val, r_val):
    try:
        mic = float(mic_value)
        s_val = float(s_val)
        r_val = float(r_val)
        if mic <= s_val:
            return 'S'
        elif mic >= r_val:
            return 'R'
        else:
            return 'I'
    except:
        return 'Data unavailable'

# Apply interpretation for each antibiotic column
for abbrev, full_name in abbrev_to_name.items():
    mic_col = f"{abbrev}_MIC"
    interp_col = f"{abbrev}_Interpretation"

    def interpret_row(row):
        organism = row['Organism']
        mic = row.get(mic_col, None)

        match = clsi_df[
            (clsi_df['Organism'].str.strip().str.lower() == str(organism).strip().lower()) &
            (clsi_df['Antibiotic'].str.strip().str.lower() == full_name.lower())
        ]
        if not match.empty:
            s_val = match['S_breakpoint'].values[0]
            r_val = match['R_breakpoint'].values[0]
            return interpret_mic(mic, s_val, r_val)
        else:
            return 'Data unavailable'

    venatorx_df[interp_col] = venatorx_df.apply(interpret_row, axis=1)

# Optionally save the output
venatorx_df.to_excel("Venatorx_MIC_interpretations.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np

# Load CLSI breakpoints with multi-row header
clsi_df = pd.read_excel("/content/CLSI_breakpoints.xlsx", sheet_name="All combined", header=[0, 1])

# Flatten the multi-index header
clsi_df.columns = [
    col[0] if col[0] != 'CLSI MIC breakpoints' else col[1]
    for col in clsi_df.columns
]

# Rename for consistency
clsi_df.rename(columns={
    'Organism name': 'Organism',
    'S': 'S_breakpoint',
    'I': 'I_breakpoint',
    'R': 'R_breakpoint'
}, inplace=True)

# Load Venatorx data
venatorx_df = pd.read_excel("/content/Venatorx surveillance data_2024_06_06.xlsx")

# Antibiotic abbreviation to full name mapping
abbrev_to_name = {
    'CAZ': 'Ceftazidime',
    'C': 'Chloramphenicol',
    'CIP': 'Ciprofloxacin',
    'CL': 'Clindamycin',
    'FEP': 'Cefepime',
    'GM': 'Gentamicin',
    'IPM': 'Imipenem',
    'LVX': 'Levofloxacin',
    'MEM': 'Meropenem',
    'MI': 'Minocycline',
    'SXT': 'Trimethoprim/Sulfamethoxazole',
    'TIM': 'Ticarcillin/Clavulanic acid',
    'TZP': 'Piperacillin/Tazobactam'
}

# MIC interpretation function with handling of blanks, dashes, and NaNs
def interpret_mic(mic_value, s_val, r_val):
    try:
        mic = float(mic_value)
        s_val = float(s_val)
        r_val = float(r_val)
        if mic <= s_val:
            return 'S'
        elif mic >= r_val:
            return 'R'
        else:
            return 'I'
    except:
        return ' - '

# Iterate through each antibiotic
for abbrev, full_name in abbrev_to_name.items():
    mic_col = f"{abbrev}_MIC"
    result_col = f"{abbrev}_Interpretation"

    def interpret_row(row):
        organism = row.get('Organism')
        mic = row.get(mic_col)

        # Check if MIC is missing or not usable
        if pd.isna(mic) or mic == '-' or str(mic).strip() == '':
            return ' - '

        # Find matching CLSI row
        match = clsi_df[
            (clsi_df['Organism'].str.strip().str.lower() == str(organism).strip().lower()) &
            (clsi_df['Antibiotic'].str.strip().str.lower() == full_name.lower())
        ]

        if not match.empty:
            s_val = match['S_breakpoint'].values[0]
            r_val = match['R_breakpoint'].values[0]
            # Check if S or R is missing or invalid
            if pd.isna(s_val) or pd.isna(r_val) or s_val == '-' or r_val == '':
                return '-'
            return interpret_mic(mic, s_val, r_val)
        else:
            return '-'

    venatorx_df[result_col] = venatorx_df.apply(interpret_row, axis=1)

# Save the final interpreted file
venatorx_df.to_excel("Venatorx_MIC_interpretations_cleaned.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
import os
import re

# Load CLSI breakpoints
clsi_df = pd.read_excel("/content/CLSI_breakpoints (1).xlsx", sheet_name="All combined", header=[0, 1])
clsi_df.columns = [col[0] if col[0] != 'CLSI MIC breakpoints' else col[1] for col in clsi_df.columns]
clsi_df.rename(columns={'Organism name': 'Organism', 'S': 'S_breakpoint', 'I': 'I_breakpoint', 'R': 'R_breakpoint'}, inplace=True)

# Updated abbreviation mappings
abbrev_to_name = {
    'AMC': 'Amoxicillin-Clavulanate', 'AMP': 'Ampicillin', 'AMX': 'Amoxicillin', 'AXO': 'Ceftriaxone',
    'AZM': 'Azithromycin', 'CDN': 'Clindamycin', 'CEC': 'Cephalexin', 'CLA': 'Clarithromycin',
    'CXM': 'Cefuroxime', 'DIN': 'Dinitrobenzamide', 'ERY': 'Erythromycin', 'FIX': 'Cefixime',
    'LEV': 'Levofloxacin', 'MXF': 'Moxifloxacin', 'PEN': 'Penicillin', 'POD': 'Cefpodoxime',
    'SXT': 'Trimethoprim/Sulfamethoxazole', 'CAZ': 'Ceftazidime', 'C': 'Chloramphenicol',
    'CIP': 'Ciprofloxacin', 'CL': 'Clindamycin', 'FEP': 'Cefepime', 'GM': 'Gentamicin',
    'IPM': 'Imipenem', 'LVX': 'Levofloxacin', 'MEM': 'Meropenem', 'MI': 'Minocycline',
    'TIM': 'Ticarcillin/Clavulanic acid', 'TZP': 'Piperacillin/Tazobactam'
}
name_to_abbrev = {v.lower(): k for k, v in abbrev_to_name.items()}

# MIC interpretation logic
def interpret_mic(mic_value, s_val, r_val):
    try:
        mic = float(mic_value)
        s_val = float(s_val)
        r_val = float(r_val)
        if mic <= s_val:
            return 'S'
        elif mic >= r_val:
            return 'R'
        else:
            return 'I'
    except:
        return 'Data unavailable'

# Extract year based on column availability
def extract_year(row):
    for col in ['Study Year', 'Year']:
        if col in row and pd.notnull(row[col]):
            return int(row[col])
    if 'Collection Date' in row and pd.notnull(row['Collection Date']):
        match = re.search(r'(\d{4})$', str(row['Collection Date']))
        if match:
            return int(match.group(1))
    if 'Date Collected' in row and pd.notnull(row['Date Collected']):
        match = re.search(r'^(\d{4})', str(row['Date Collected']))
        if match:
            return int(match.group(1))
    return None

# Main processing function
def process_dataset(file_path):
    df = pd.read_excel(file_path)

    # Detect organism column
    org_col = 'Organism' if 'Organism' in df.columns else 'Organism Name' if 'Organism Name' in df.columns else None
    if not org_col:
        raise ValueError("No organism column found.")

    # Try to detect other columns
    gender_col = next((col for col in df.columns if 'gender' in col.lower()), None)
    age_col = next((col for col in df.columns if 'age' in col.lower()), None)
    country_col = next((col for col in df.columns if 'country' in col.lower()), None)

    # Extract year using flexible rules
    df['Year_collected'] = df.apply(extract_year, axis=1)

    # Prepare output structure
    output = pd.DataFrame()
    output['Organism'] = df[org_col]
    if country_col: output['Country'] = df[country_col]
    output['Year collected'] = df['Year_collected']
    if gender_col: output['Gender'] = df[gender_col]
    if age_col: output['Age'] = df[age_col]

    for abbrev, full_name in abbrev_to_name.items():
        mic_col = f"{abbrev}_MIC"
        if mic_col not in df.columns:
            continue

        def interpret_row(row):
            organism = row[org_col]
            mic = row[mic_col]
            match = clsi_df[
                (clsi_df['Organism'].str.strip().str.lower() == str(organism).strip().lower()) &
                (
                    (clsi_df['Antibiotic'].str.strip().str.lower() == full_name.lower()) |
                    (clsi_df['Antibiotic'].str.strip().str.lower() == abbrev.lower())
                )
            ]
            if not match.empty:
                s_val = match['S_breakpoint'].values[0]
                r_val = match['R_breakpoint'].values[0]
                return interpret_mic(mic, s_val, r_val)
            return 'Data unavailable'

        output[f'{full_name} (MIC Interpretation)'] = df.apply(interpret_row, axis=1)

    # Save result
    output_filename = os.path.splitext(os.path.basename(file_path))[0] + "_MIC_Analysis.xlsx"
    output.to_excel(output_filename, index=False)
    print(f" Saved: {output_filename}")

# ========= USAGE =========

dataset_paths = [
    "/content/Venatorx surveillance data_2024_06_06 (1).xlsx",
    "/content/GSK_SOAR_201910 raw data (2).xlsx",
    "/content/Omadacycline_2014_to_2023_Surveillance_data (1).xlsx",
    "/content/Updated_Shionogi Five year SIDERO-WT Surveillance data(without strain number)_Vivli_220409 (1).xlsx"
]

for path in dataset_paths:
    process_dataset(path)


 Saved: Venatorx surveillance data_2024_06_06 (1)_MIC_Analysis.xlsx
 Saved: GSK_SOAR_201910 raw data (2)_MIC_Analysis.xlsx
 Saved: Omadacycline_2014_to_2023_Surveillance_data (1)_MIC_Analysis.xlsx
 Saved: Updated_Shionogi Five year SIDERO-WT Surveillance data(without strain number)_Vivli_220409 (1)_MIC_Analysis.xlsx


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files

# Upload up to 4 Excel files
uploaded = files.upload()

# Prepare file paths
uploaded_files = list(uploaded.keys())
uploaded_paths = [os.path.join("/content", fname) for fname in uploaded_files]

# Antibiotic abbreviation mapping
abbrev_to_name = {
    'AMC': 'Amoxicillin-Clavulanate', 'AMP': 'Ampicillin', 'AMX': 'Amoxicillin', 'AXO': 'Ceftriaxone',
    'AZM': 'Azithromycin', 'CDN': 'Clindamycin', 'CEC': 'Cephalexin', 'CLA': 'Clarithromycin',
    'CXM': 'Cefuroxime', 'DIN': 'Dinitrobenzamide', 'ERY': 'Erythromycin', 'FIX': 'Cefixime',
    'LEV': 'Levofloxacin', 'MXF': 'Moxifloxacin', 'PEN': 'Penicillin', 'POD': 'Cefpodoxime',
    'SXT': 'Trimethoprim-Sulfamethoxazole', 'CAZ': 'Ceftazidime', 'C': 'Chloramphenicol',
    'CIP': 'Ciprofloxacin', 'CL': 'Clindamycin', 'FEP': 'Cefepime', 'GM': 'Gentamicin',
    'IPM': 'Imipenem', 'LVX': 'Levofloxacin', 'MEM': 'Meropenem', 'MI': 'Minocycline',
    'TIM': 'Ticarcillin/Clavulanic acid', 'TZP': 'Piperacillin/Tazobactam'
}
name_to_abbrev = {v.lower(): k for k, v in abbrev_to_name.items()}

# Extract year helper
def extract_year(row):
    for col in ['Study Year', 'Year', 'Collection Date', 'Date Collected']:
        if col in row:
            val = str(row[col])
            digits = ''.join(filter(str.isdigit, val))
            if len(digits) >= 4:
                if col == 'Collection Date':
                    return int(digits[-4:])
                elif col == 'Date Collected':
                    return int(digits[:4])
                else:
                    return int(digits[:4])
    return np.nan


def analyze_dataset(path):
    df = pd.read_excel(path)
    dataset_name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n\n====== Analyzing: {dataset_name} ======\n")

    df['Year'] = df.apply(extract_year, axis=1)


    if 'Organism Name' in df.columns:
        df['Organism'] = df['Organism Name']
    elif 'Organism' not in df.columns:
        df['Organism'] = 'Unknown'

    if 'Country' not in df.columns:
        df['Country'] = 'Unknown'


    age_cols = [col for col in df.columns if 'age' in col.lower()]
    gender_cols = [col for col in df.columns if 'gender' in col.lower()]
    selected_cols = ['Organism', 'Country', 'Year'] + age_cols + gender_cols

    print("Generating missing data heatmap...")
    plt.figure(figsize=(12, 6))
    sns.heatmap(df[selected_cols].isnull(), cbar=False, cmap='viridis')
    plt.title(f"Missing Data Heatmap: {dataset_name}")
    plt.tight_layout()
    plt.show()


    print("Plotting frequency by year...")
    plt.figure(figsize=(10, 4))
    sns.countplot(x='Year', data=df)
    plt.title(f"Frequency by Year: {dataset_name}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


    print("Plotting country frequency (horizontal)...")
    plt.figure(figsize=(12, 4))
    top_countries = df['Country'].value_counts().head(15).index
    sns.countplot(data=df[df['Country'].isin(top_countries)], y='Country', order=top_countries)
    plt.title(f"Top 15 Countries by Sample Count (Horizontal): {dataset_name}")
    plt.tight_layout()
    plt.show()


    print("Plotting country frequency (vertical)...")
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df[df['Country'].isin(top_countries)], x='Country', order=top_countries)
    plt.title(f"Top 15 Countries by Sample Count (Vertical): {dataset_name}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


    print("Antibiotic usage summary:")
    ab_counts = {}
    for col in df.columns:
        for abbr, fullname in abbrev_to_name.items():
            if abbr in col or fullname.lower() in col.lower():
                ab_counts[fullname] = ab_counts.get(fullname, 0) + df[col].notna().sum()

    ab_table = pd.DataFrame(list(ab_counts.items()), columns=['Antibiotic', 'Count']).sort_values(by='Count', ascending=False)
    print(ab_table)

    # Plot usage
    plt.figure(figsize=(12, 6))
    sns.barplot(data=ab_table, x='Count', y='Antibiotic', palette='magma')
    plt.title(f"Antibiotic Usage Count: {dataset_name}")
    plt.tight_layout()
    plt.show()

  -
    excel_output = f"/content/{dataset_name}_AMR_Visual_Report.xlsx"
    with pd.ExcelWriter(excel_output, engine='xlsxwriter') as writer:
        df[selected_cols].to_excel(writer, index=False, sheet_name='Metadata')
        ab_table.to_excel(writer, index=False, sheet_name='Antibiotic Usage')
        df['Year'].value_counts().reset_index().rename(columns={'index': 'Year', 'Year': 'Count'}).to_excel(writer, index=False, sheet_name='Year Frequency')
        df['Country'].value_counts().reset_index().rename(columns={'index': 'Country', 'Country': 'Count'}).to_excel(writer, index=False, sheet_name='Country Frequency')

    print(f"\n Excel report saved: {excel_output}")


for path in uploaded_paths:
    analyze_dataset(path)
